In [0]:
data = [
    (101, "Roshan", "IT", 65000),
    (102, "Rahul", "HR", 40000),
    (103, "Priya", "IT", 75000),
    (104, "Amit", "Finance", 55000),
    (105, "Neha", "HR", 48000),
    (106, "Vijay", "Finance", 82000)
]

columns = ["emp_id", "name", "department", "salary"]

df = spark.createDataFrame(data,columns)
df.show()

In [0]:
from pyspark.sql.functions import *
df1=df.filter(df.department)=="IT"
df_result = df.select("name","salary").orderBy(
    "salary",asecending = True
)
df_result.show()

Calculate the average salary for each department and display only departments whose average salary is greater than 55000.



In [0]:
from pyspark.sql.functions import col , avg 
df_grouped = df.groupBy("department").agg(
    avg('salary').alias("avrage_salary")
)
df_fillterd = df_grouped.filter(
    col("avrage_salary") > 55000
)
result = df_fillterd.orderBy(
    col("avrage_salary").desc()

)
result.show()



Find the highest-paid employee in each department

In [0]:
from pyspark.sql.functions import col ,dense_rank
from pyspark.sql.window import Window
window_spect = Window.partitionBy("department").orderBy(
    (col('salary').desc())
)

df_fillterd = df.withColumn(
    "rank",
    dense_rank().over(window_spect)
)
df_result = df_fillterd.filter(col('rank')==1)
result = df_result.select(
    "department", "name","salary","rank"
)
result.show()


In [0]:
employees = [
    (101, "Roshan", 10, 65000),
    (102, "Rahul", 20, 45000),
    (103, "Priya", 10, 75000),
    (104, "Amit", 30, 55000),
    (105, "Neha", None, 48000),
    (106, "Vijay", 40, 82000)
]

employee_columns = ["emp_id", "name", "dept_id", "salary"]

emp_df = spark.createDataFrame(employees)

departments = [
    (10, "IT"),
    (20, "HR"),
    (30, "Finance"),
    (50, "Marketing")
]

department_columns = ["dept_id", "department"]
dept_df = spark.createDataFrame(departments,department_columns)
dept_df.show()

In [0]:
employees = [
    (101, "Roshan", 10, 65000),
    (102, "Rahul", 20, 45000),
    (103, "Priya", 10, 75000),
    (104, "Amit", 30, 55000),
    (105, "Neha", None, 48000),
    (106, "Vijay", 40, 82000)
]

employee_columns = ["emp_id", "name", "dept_id", "salary"]

emp_df = spark.createDataFrame(
    employees,
    employee_columns
)

emp_df.printSchema()
emp_df.show()

Create both DataFrames.



Perform a left join using dept_id.



Calculate the average salary for each department.



Keep departments whose average salary is greater than 50000.



Display department and average_salary in descending order of average salary.

In [0]:


from pyspark.sql.functions import col ,avg 
df_joined = emp_df.join(
    dept_df,
    on = "dept_id",
    how = "left"    
)
#calculaate the avrage salary as per department 
df_group = df_joined.groupBy("department").agg(
    avg("salary").alias("avrage_salary")
)
df_fillterd = df_group.filter(col("avrage_salary")> 50000)
result =df_fillterd.orderBy(
    col("avrage_salary").desc()
)
result.show()


In [0]:
dept_df.printSchema()
dept_df.show()

Create a window partitioned by department.



Order rows by emp_id in ascending order.



Calculate the running total of salary within each department.



Store the result in a new column named running_total.



Display emp_id, name, department, salary, and running_total.



Sort the final result by department and emp_id.

In [0]:
from pyspark.sql.functions import col , avg ,sum
from pyspark.sql.window import Window

window_spect = Window.partitionBy("department").orderBy(
    col('emp_id').asc()

).rowsBetween(
    Window.unboundedPreceding,
    Window.currentRow 
    
)

df1= df.withColumn(
    "runing_total",
    sum('salary').over(window_spect)
)
result = df1.select(
    "name","department","salary","runing_total"
).orderBy(
    col("department").asc(),
    col('emp_id').asc()
)

result.show()


In [0]:
import requests
# reas data 
responses = requests.get('https://raw.githubusercontent.com/yashvirout2022/pyspark/refs/heads/main/customers.csv')
# save data 
with open (destination_path,wb ) as f 
f.write (responses,conent)

In [0]:

# read dat from landing folder 
spark.read.format("csv").option("header",True)\
    .load('https://raw.githubusercontent.com/yashvirout2022/pyspark/refs/heads/main/customers.csv')
df = df.withColumn('ingetion_timestamp',current_timestamp())\
    .withColumn("source",lit('github'))

df.write \
    .format("parquet") \
    .mode("overwrite") \
    .save()